In [23]:
# importsss
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

# Solar position calculation
from pysolar.solar import get_altitude, get_azimuth


In [24]:
# paths..
PROCESSED_DATA_PATH = Path(r"C:\Rainbow\data\processed")
FEATURES_OUTPUT_PATH = Path(r"C:\Rainbow\data\processed\features")
FIGURES_PATH = Path(r"C:\Rainbow\app\results\figures")

# Create directories
FEATURES_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

In [25]:
input_file = PROCESSED_DATA_PATH / "C:\Rainbow\data\processed\combined_nasa_power_data.csv"
print(f"\nLoading data from: {input_file}")

df = pd.read_csv(input_file)
print(f"[DONE] Loaded {len(df):,} records")
print(f"       Columns: {len(df.columns)}")
print(f"       Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())


Loading data from: C:\Rainbow\data\processed\combined_nasa_power_data.csv
[DONE] Loaded 1,490,832 records
       Columns: 16
       Memory: 326.09 MB

First 5 rows:
   YEAR  MO  DY  HR  ALLSKY_SFC_SW_DWN  CLRSKY_SFC_SW_DWN  ALLSKY_KT   SZA  \
0  2020   1   1   0                0.0                0.0     -999.0  90.0   
1  2020   1   1   1                0.0                0.0     -999.0  90.0   
2  2020   1   1   2                0.0                0.0     -999.0  90.0   
3  2020   1   1   3                0.0                0.0     -999.0  90.0   
4  2020   1   1   4                0.0                0.0     -999.0  90.0   

   PRECTOTCORR   RH2M    T2M  WS2M  WD2M      PS      city   file_source  
0          0.0  91.45  16.48  1.07  46.9  101.48  Agartala  Agartala.csv  
1          0.0  94.17  16.06  1.09  54.0  101.45  Agartala  Agartala.csv  
2          0.0  95.32  15.85  1.12  60.4  101.42  Agartala  Agartala.csv  
3          0.0  95.11  15.85  1.09  67.9  101.38  Agartala  Agart

In [26]:
print("\nCombining YEAR, MO, DY, HR into datetime...")
df['datetime'] = pd.to_datetime(df[['YEAR', 'MO', 'DY', 'HR']].rename(
    columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}
))
print("[DONE] Created datetime column")

# Extract basic time features
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour
df['day_of_year'] = df['datetime'].dt.dayofyear
df['day_of_week'] = df['datetime'].dt.dayofweek
df['week_of_year'] = df['datetime'].dt.isocalendar().week

print("[DONE] Extracted time components")


Combining YEAR, MO, DY, HR into datetime...
[DONE] Created datetime column
[DONE] Extracted time components


In [27]:
def get_season(month):
    """
    Get season for Indian climate
    - Winter: December, January, February
    - Spring: March, April, May
    - Monsoon: June, July, August, September
    - Autumn: October, November
    """
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:  # 10, 11
        return 'Autumn'

df['season'] = df['month'].apply(get_season)
print("[DONE] Created season feature")

# Create season encoding
season_mapping = {'Winter': 0, 'Spring': 1, 'Monsoon': 2, 'Autumn': 3}
df['season_encoded'] = df['season'].map(season_mapping)

print("\nSeason distribution:")
print(df['season'].value_counts().sort_index())

[DONE] Created season feature

Season distribution:
season
Autumn     248880
Monsoon    497760
Spring     375360
Winter     368832
Name: count, dtype: int64


In [28]:
def get_time_of_day(hour):
    """
    Categorize hour into time of day
    Rainbows are most common during:
    - Early Morning (5-9 AM)
    - Late Afternoon/Evening (4-7 PM)
    """
    if 5 <= hour <= 9:
        return 'Early_Morning'
    elif 10 <= hour <= 15:
        return 'Midday'
    elif 16 <= hour <= 19:
        return 'Late_Afternoon'
    elif 20 <= hour <= 23:
        return 'Night'
    else:  # 0-4
        return 'Late_Night'

df['time_of_day'] = df['hour'].apply(get_time_of_day)
print("[DONE] Created time_of_day feature")

# Rainbow favorable time (early morning or late afternoon)
df['rainbow_favorable_time'] = df['time_of_day'].isin(['Early_Morning', 'Late_Afternoon']).astype(int)

print("\nTime of day distribution:")
print(df['time_of_day'].value_counts().sort_index())
print(f"\nRainbow favorable times: {df['rainbow_favorable_time'].sum():,} records ({df['rainbow_favorable_time'].mean()*100:.1f}%)")

[DONE] Created time_of_day feature

Time of day distribution:
time_of_day
Early_Morning     310590
Late_Afternoon    248472
Late_Night        310590
Midday            372708
Night             248472
Name: count, dtype: int64

Rainbow favorable times: 559,062 records (37.5%)


In [29]:
city_coordinates = {
    'Agartala': (23.8315, 91.2868),
    'Aizwal': (23.7307, 92.7173),
    'Amravati': (16.5062, 80.6480),
    'Bengaluru': (12.9716, 77.5946),
    'Bhopal': (23.2599, 77.4126),
    'Bhubaneswar': (20.2961, 85.8245),
    'Chandigarh': (30.7333, 76.7794),
    'Chennai': (13.0827, 80.2707),
    'Daman': (20.4283, 72.8397),
    'Dehradun': (30.3165, 78.0322),
    'Dispur': (26.1445, 91.7362),
    'Gandhinagar': (23.2156, 72.6369),
    'Gangtok': (27.3389, 88.6065),
    'Hyderabad': (17.3850, 78.4867),
    'Imphal': (24.8170, 93.9368),
    'Itanagar': (27.0844, 93.6053),
    'Jaipur': (26.9124, 75.7873),
    'Kavaratti': (10.5667, 72.6417),
    'Kohima': (25.6747, 94.1103),
    'Kolkata': (22.5726, 88.3639),
    'Leh': (34.1526, 77.5771),
    'Lucknow': (26.8467, 80.9462),
    'Mumbai': (19.0760, 72.8777),
    'New_Delhi': (28.6139, 77.2090),
    'Panaji': (15.4909, 73.8278),
    'Patna': (25.5941, 85.1376),
    'Port_Blair': (11.6234, 92.7265),
    'Puducherry': (11.9416, 79.8083),
    'Raipur': (21.2514, 81.6296),
    'Ranchi': (23.3441, 85.3096),
    'Shillong': (25.5788, 91.8933),
    'Shimla': (31.1048, 77.1734),
    'Srinagar': (34.0837, 74.7973),
    'Thiruvananthapuram': (8.5241, 76.9366)
}

# Add latitude and longitude to dataframe
df['latitude'] = df['city'].map(lambda x: city_coordinates.get(x, (None, None))[0])
df['longitude'] = df['city'].map(lambda x: city_coordinates.get(x, (None, None))[1])

missing_coords = df[df['latitude'].isnull()]['city'].unique()
if len(missing_coords) > 0:
    print(f"[WARNING] Missing coordinates for: {missing_coords}")
else:
    print("[DONE] All cities have coordinates")


[DONE] All cities have coordinates


In [30]:
def calculate_solar_position(row):
    """Calculate solar altitude and azimuth"""
    try:
        # Convert to timezone-aware datetime (UTC)
        dt = row['datetime'].replace(tzinfo=timezone.utc)
        lat = row['latitude']
        lon = row['longitude']
        
        altitude = get_altitude(lat, lon, dt)
        azimuth = get_azimuth(lat, lon, dt)
        
        return pd.Series({
            'solar_altitude': altitude,
            'solar_azimuth': azimuth
        })
    except:
        return pd.Series({
            'solar_altitude': np.nan,
            'solar_azimuth': np.nan
        })

# Calculate solar position (this takes time!)
print("Calculating solar altitude and azimuth...")
print("Processing in batches...")

# Process in chunks to show progress
chunk_size = 100000
num_chunks = len(df) // chunk_size + 1
solar_data = []

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(df))
    
    chunk = df.iloc[start_idx:end_idx]
    chunk_solar = chunk.apply(calculate_solar_position, axis=1)
    solar_data.append(chunk_solar)
    
    print(f"  Processed {end_idx:,} / {len(df):,} records ({end_idx/len(df)*100:.1f}%)")

# Combine all chunks
solar_df = pd.concat(solar_data, ignore_index=True)
df['solar_altitude'] = solar_df['solar_altitude']
df['solar_azimuth'] = solar_df['solar_azimuth']

print("[DONE] Solar position calculated")
print(f"  Solar altitude range: {df['solar_altitude'].min():.2f} to {df['solar_altitude'].max():.2f} degrees")

Calculating solar altitude and azimuth...
Processing in batches...
  Processed 100,000 / 1,490,832 records (6.7%)
  Processed 200,000 / 1,490,832 records (13.4%)
  Processed 300,000 / 1,490,832 records (20.1%)
  Processed 400,000 / 1,490,832 records (26.8%)
  Processed 500,000 / 1,490,832 records (33.5%)
  Processed 600,000 / 1,490,832 records (40.2%)
  Processed 700,000 / 1,490,832 records (47.0%)
  Processed 800,000 / 1,490,832 records (53.7%)
  Processed 900,000 / 1,490,832 records (60.4%)
  Processed 1,000,000 / 1,490,832 records (67.1%)
  Processed 1,100,000 / 1,490,832 records (73.8%)
  Processed 1,200,000 / 1,490,832 records (80.5%)
  Processed 1,300,000 / 1,490,832 records (87.2%)
  Processed 1,400,000 / 1,490,832 records (93.9%)
  Processed 1,490,832 / 1,490,832 records (100.0%)
[DONE] Solar position calculated
  Solar altitude range: -89.86 to 89.61 degrees


In [31]:
# 1. Sun is visible (altitude > 0)
df['sun_visible'] = (df['solar_altitude'] > 0).astype(int)

# 2. Optimal solar altitude for rainbows (typically 0-42 degrees)
df['optimal_sun_angle'] = ((df['solar_altitude'] > 0) & (df['solar_altitude'] < 42)).astype(int)

# 3. Rain present
df['rain_present'] = (df['PRECTOTCORR'] > 0).astype(int)

# 4. Light rain (better for rainbows than heavy rain)
df['light_rain'] = ((df['PRECTOTCORR'] > 0) & (df['PRECTOTCORR'] < 10)).astype(int)

# 5. Moderate cloud cover (not clear sky, not completely overcast)
# Using ALLSKY_KT (clearness index): 0.3-0.7 is ideal
df['moderate_clouds'] = ((df['ALLSKY_KT'] > 0.3) & (df['ALLSKY_KT'] < 0.7)).astype(int)

# 6. Sun opposite to rain direction (simplified - when sun is low and rain present)
df['sun_rain_geometry'] = ((df['solar_altitude'] > 0) & 
                            (df['solar_altitude'] < 42) & 
                            (df['PRECTOTCORR'] > 0)).astype(int)

# 7. High humidity (supports water droplets)
df['high_humidity'] = (df['RH2M'] > 60).astype(int)

# 8. Rainbow favorability score (0-10)
df['rainbow_score'] = (
    df['rainbow_favorable_time'] * 2 +  # Time of day (2 points)
    df['optimal_sun_angle'] * 2 +       # Sun angle (2 points)
    df['light_rain'] * 2 +              # Light rain (2 points)
    df['moderate_clouds'] * 2 +         # Cloud cover (2 points)
    df['high_humidity'] * 1 +           # Humidity (1 point)
    df['sun_rain_geometry'] * 1         # Geometry (1 point)
)

print("[DONE] Created rainbow-specific features")
print("\nRainbow Feature Statistics:")
print(f"  Sun visible: {df['sun_visible'].sum():,} records ({df['sun_visible'].mean()*100:.1f}%)")
print(f"  Optimal sun angle: {df['optimal_sun_angle'].sum():,} records ({df['optimal_sun_angle'].mean()*100:.1f}%)")
print(f"  Rain present: {df['rain_present'].sum():,} records ({df['rain_present'].mean()*100:.1f}%)")
print(f"  All favorable conditions: {df[df['rainbow_score'] >= 8].shape[0]:,} records ({df[df['rainbow_score'] >= 8].shape[0]/len(df)*100:.2f}%)")

[DONE] Created rainbow-specific features

Rainbow Feature Statistics:
  Sun visible: 752,700 records (50.5%)
  Optimal sun angle: 433,889 records (29.1%)
  Rain present: 822,750 records (55.2%)
  All favorable conditions: 48,410 records (3.25%)


In [32]:
# Temperature categories
df['temp_category'] = pd.cut(df['T2M'], 
                              bins=[-np.inf, 10, 20, 30, np.inf],
                              labels=['Cold', 'Cool', 'Warm', 'Hot'])

# Wind speed categories
df['wind_category'] = pd.cut(df['WS2M'],
                              bins=[-np.inf, 2, 5, 10, np.inf],
                              labels=['Calm', 'Light', 'Moderate', 'Strong'])

# Precipitation categories
df['precip_category'] = pd.cut(df['PRECTOTCORR'],
                                bins=[-0.01, 0, 1, 5, np.inf],
                                labels=['None', 'Light', 'Moderate', 'Heavy'])

print("[DONE] Created weather categories")

[DONE] Created weather categories


In [33]:
new_features = [
    'datetime', 'month', 'day', 'hour', 'day_of_year', 'day_of_week',
    'season', 'season_encoded', 'time_of_day', 'rainbow_favorable_time',
    'latitude', 'longitude', 'solar_altitude', 'solar_azimuth',
    'sun_visible', 'optimal_sun_angle', 'rain_present', 'light_rain',
    'moderate_clouds', 'sun_rain_geometry', 'high_humidity', 'rainbow_score',
    'temp_category', 'wind_category', 'precip_category'
]

print(f"\nOriginal features: {len(df.columns) - len(new_features)}")
print(f"New features created: {len(new_features)}")
print(f"Total features: {len(df.columns)}")

print("\nNew Feature List:")
for i, feat in enumerate(new_features, 1):
    print(f"  {i:2d}. {feat}")


Original features: 18
New features created: 25
Total features: 43

New Feature List:
   1. datetime
   2. month
   3. day
   4. hour
   5. day_of_year
   6. day_of_week
   7. season
   8. season_encoded
   9. time_of_day
  10. rainbow_favorable_time
  11. latitude
  12. longitude
  13. solar_altitude
  14. solar_azimuth
  15. sun_visible
  16. optimal_sun_angle
  17. rain_present
  18. light_rain
  19. moderate_clouds
  20. sun_rain_geometry
  21. high_humidity
  22. rainbow_score
  23. temp_category
  24. wind_category
  25. precip_category


In [34]:
# Figure 1: Rainbow Score Distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Rainbow score distribution
axes[0, 0].hist(df['rainbow_score'], bins=11, range=(-0.5, 10.5), 
                color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Rainbow Favorability Score', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of Rainbow Favorability Score', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)
axes[0, 0].axvline(x=8, color='red', linestyle='--', label='High Favorability Threshold')
axes[0, 0].legend()

# Plot 2: Solar altitude distribution
sun_data = df[df['sun_visible'] == 1]['solar_altitude']
axes[0, 1].hist(sun_data, bins=50, color='orange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Solar Altitude (degrees)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Solar Altitude Distribution (When Sun is Visible)', fontsize=12, fontweight='bold')
axes[0, 1].axvline(x=42, color='red', linestyle='--', label='Rainbow Optimal Limit (42°)')
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].legend()

# Plot 3: Rainbow favorable conditions by time of day
time_rainbow = df.groupby('time_of_day')['rainbow_score'].mean().sort_values(ascending=False)
axes[1, 0].barh(range(len(time_rainbow)), time_rainbow.values, color='lightgreen')
axes[1, 0].set_yticks(range(len(time_rainbow)))
axes[1, 0].set_yticklabels(time_rainbow.index)
axes[1, 0].set_xlabel('Average Rainbow Score', fontsize=11)
axes[1, 0].set_title('Rainbow Favorability by Time of Day', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Plot 4: Rainbow favorable conditions by season
season_rainbow = df.groupby('season')['rainbow_score'].mean().sort_values(ascending=False)
axes[1, 1].bar(range(len(season_rainbow)), season_rainbow.values, color='lightcoral')
axes[1, 1].set_xticks(range(len(season_rainbow)))
axes[1, 1].set_xticklabels(season_rainbow.index, rotation=0)
axes[1, 1].set_ylabel('Average Rainbow Score', fontsize=11)
axes[1, 1].set_title('Rainbow Favorability by Season', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'rainbow_feature_analysis.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: rainbow_feature_analysis.png")

# Figure 2: Feature correlations
fig, ax = plt.subplots(figsize=(14, 10))

# Select numeric features for correlation
numeric_features = [
    'PRECTOTCORR', 'RH2M', 'T2M', 'WS2M', 'ALLSKY_KT',
    'solar_altitude', 'rainbow_score', 'hour', 'month'
]

corr_matrix = df[numeric_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'feature_correlations.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: feature_correlations.png")

plt.close('all')

[DONE] Saved: rainbow_feature_analysis.png
[DONE] Saved: feature_correlations.png


In [35]:
# Save full dataset with all features
output_file = FEATURES_OUTPUT_PATH / "rainbow_data_with_features.csv"
df.to_csv(output_file, index=False, encoding='utf-8')
print(f"[DONE] Saved enhanced dataset: {output_file}")
print(f"       Size: {output_file.stat().st_size / (1024**2):.2f} MB")
print(f"       Records: {len(df):,}")
print(f"       Features: {len(df.columns)}")

# Save a subset of highly favorable rainbow conditions for easier labeling
rainbow_candidates = df[df['rainbow_score'] >= 8].copy()
candidates_file = FEATURES_OUTPUT_PATH / "rainbow_favorable_candidates.csv"
rainbow_candidates.to_csv(candidates_file, index=False, encoding='utf-8')
print(f"\n[DONE] Saved rainbow candidate records: {candidates_file}")
print(f"       Records: {len(rainbow_candidates):,} ({len(rainbow_candidates)/len(df)*100:.2f}% of total)")
print(f"       These are the MOST LIKELY times for rainbows!")

# Save feature documentation
feature_doc = """
FEATURE DOCUMENTATION - Rainbow Prediction Project
================================================================================

ORIGINAL NASA POWER FEATURES:
1. YEAR, MO, DY, HR - Time components
2. ALLSKY_SFC_SW_DWN - All Sky Surface Shortwave Downward Irradiance (W/m²)
3. CLRSKY_SFC_SW_DWN - Clear Sky Surface Shortwave Downward Irradiance (W/m²)
4. ALLSKY_KT - All Sky Insolation Clearness Index (0-1)
5. SZA - Solar Zenith Angle (degrees)
6. PRECTOTCORR - Precipitation Corrected (mm/day)
7. RH2M - Relative Humidity at 2 Meters (%)
8. T2M - Temperature at 2 Meters (°C)
9. WS2M - Wind Speed at 2 Meters (m/s)
10. WD2M - Wind Direction at 2 Meters (degrees)
11. PS - Surface Pressure (kPa)

ENGINEERED FEATURES:

TIME FEATURES:
- datetime: Combined datetime object
- month, day, hour: Extracted time components
- day_of_year, day_of_week, week_of_year: Calendar features
- season: Winter/Spring/Monsoon/Autumn (Indian climate)
- season_encoded: Numeric encoding of season (0-3)
- time_of_day: Early_Morning/Midday/Late_Afternoon/Night/Late_Night
- rainbow_favorable_time: 1 if Early_Morning or Late_Afternoon, else 0

LOCATION FEATURES:
- city: City name
- latitude, longitude: Geographic coordinates

SOLAR POSITION FEATURES:
- solar_altitude: Sun's altitude angle (degrees above horizon)
- solar_azimuth: Sun's azimuth angle (degrees from north)
- sun_visible: 1 if solar_altitude > 0, else 0
- optimal_sun_angle: 1 if solar_altitude between 0-42°, else 0

RAINBOW-SPECIFIC FEATURES:
- rain_present: 1 if PRECTOTCORR > 0, else 0
- light_rain: 1 if precipitation between 0-10 mm/day, else 0
- moderate_clouds: 1 if clearness index between 0.3-0.7, else 0
- sun_rain_geometry: 1 if optimal sun angle AND rain present, else 0
- high_humidity: 1 if RH2M > 60%, else 0
- rainbow_score: Composite score (0-10) indicating rainbow favorability

DERIVED WEATHER FEATURES:
- temp_category: Cold/Cool/Warm/Hot
- wind_category: Calm/Light/Moderate/Strong
- precip_category: None/Light/Moderate/Heavy

================================================================================
RAINBOW SCORE CALCULATION:
- Rainbow favorable time: 2 points
- Optimal sun angle: 2 points
- Light rain: 2 points
- Moderate clouds: 2 points
- High humidity: 1 point
- Sun-rain geometry: 1 point
Maximum score: 10 points

INTERPRETATION:
- Score 0-3: Very unlikely for rainbow
- Score 4-6: Possible but not ideal
- Score 7-8: Good conditions for rainbow
- Score 9-10: Excellent conditions for rainbow
================================================================================
"""

doc_file = FEATURES_OUTPUT_PATH / "feature_documentation.txt"
with open(doc_file, 'w', encoding='utf-8') as f:
    f.write(feature_doc)
print(f"\n[DONE] Saved feature documentation: {doc_file}")


[DONE] Saved enhanced dataset: C:\Rainbow\data\processed\features\rainbow_data_with_features.csv
       Size: 338.77 MB
       Records: 1,490,832
       Features: 43

[DONE] Saved rainbow candidate records: C:\Rainbow\data\processed\features\rainbow_favorable_candidates.csv
       Records: 48,410 (3.25% of total)
       These are the MOST LIKELY times for rainbows!

[DONE] Saved feature documentation: C:\Rainbow\data\processed\features\feature_documentation.txt


In [36]:
summary = f"""
Total Records: {len(df):,}
Total Features: {len(df.columns)}
New Features Created: {len(new_features)}

KEY INSIGHTS:
1. Rainbow Favorable Conditions:
   - Records with score >= 8: {len(rainbow_candidates):,} ({len(rainbow_candidates)/len(df)*100:.2f}%)
   - Average rainbow score: {df['rainbow_score'].mean():.2f}
   
2. Solar Position:
   - Sun visible: {df['sun_visible'].sum():,} records ({df['sun_visible'].mean()*100:.1f}%)
   - Optimal sun angle: {df['optimal_sun_angle'].sum():,} records ({df['optimal_sun_angle'].mean()*100:.1f}%)
   
3. Weather Conditions:
   - Rain present: {df['rain_present'].sum():,} records ({df['rain_present'].mean()*100:.1f}%)
   - Light rain: {df['light_rain'].sum():,} records ({df['light_rain'].mean()*100:.1f}%)
   - Moderate clouds: {df['moderate_clouds'].sum():,} records ({df['moderate_clouds'].mean()*100:.1f}%)

4. Best Time for Rainbows:
   - Top season: {season_rainbow.index[0]} (avg score: {season_rainbow.values[0]:.2f})
   - Top time of day: {time_rainbow.index[0]} (avg score: {time_rainbow.values[0]:.2f})


"""

print(summary)

print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE!")
print("=" * 80)
print("\nFiles created:")
print(f"1. Enhanced dataset: {output_file}")
print(f"2. Rainbow candidates: {candidates_file}")
print(f"3. Feature documentation: {doc_file}")
print(f"4. Visualizations: {FIGURES_PATH}/")
print("\nYou can now proceed to:")
print("1. Review the rainbow_favorable_candidates.csv (only 0.32% of data!)")
print("2. Start rainbow labeling on these high-probability records")
print("3. Perform Exploratory Data Analysis")
print("=" * 80)


Total Records: 1,490,832
Total Features: 43
New Features Created: 25

KEY INSIGHTS:
1. Rainbow Favorable Conditions:
   - Records with score >= 8: 48,410 (3.25%)
   - Average rainbow score: 3.71

2. Solar Position:
   - Sun visible: 752,700 records (50.5%)
   - Optimal sun angle: 433,889 records (29.1%)

3. Weather Conditions:
   - Rain present: 822,750 records (55.2%)
   - Light rain: 642,850 records (43.1%)
   - Moderate clouds: 500,966 records (33.6%)

4. Best Time for Rainbows:
   - Top season: Monsoon (avg score: 4.36)
   - Top time of day: Early_Morning (avg score: 5.12)




FEATURE ENGINEERING COMPLETE!

Files created:
1. Enhanced dataset: C:\Rainbow\data\processed\features\rainbow_data_with_features.csv
2. Rainbow candidates: C:\Rainbow\data\processed\features\rainbow_favorable_candidates.csv
3. Feature documentation: C:\Rainbow\data\processed\features\feature_documentation.txt
4. Visualizations: C:\Rainbow\app\results\figures/

You can now proceed to:
1. Review the rainbow_fa